In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
import warnings
warnings.filterwarnings('ignore')

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow version: 2.10.0
GPU available: True


In [3]:
with open('dataset_stats.json', 'r') as f:
    stats = json.load(f)

mean = np.array(stats['mean'])
std = np.array(stats['std'])
num_classes = stats['num_classes']
img_size = stats['img_size']
idx_to_breed = {int(k): v for k, v in stats['idx_to_breed'].items()}
class_weight_dict = {int(k): v for k, v in stats['class_weight_dict'].items()}

print(f"✓ Number of classes: {num_classes}")
print(f"✓ Image size: {img_size}x{img_size}")

train_df = pd.read_csv('train_split.csv')
val_df = pd.read_csv('val_split.csv')
test_df = pd.read_csv('test_split.csv')

print(f"✓ Train: {len(train_df)} images")
print(f"✓ Val: {len(val_df)} images")
print(f"✓ Test: {len(test_df)} images")


✓ Number of classes: 120
✓ Image size: 224x224
✓ Train: 7155 images
✓ Val: 1533 images
✓ Test: 1534 images


In [4]:
BASE_PATH = '../dog-breed-identification/dog-breed-identification/'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')

train_df['filepath'] = train_df['id'].apply(lambda x: os.path.join(TRAIN_PATH, f"{x}.jpg"))
val_df['filepath'] = val_df['id'].apply(lambda x: os.path.join(TRAIN_PATH, f"{x}.jpg"))
test_df['filepath'] = test_df['id'].apply(lambda x: os.path.join(TRAIN_PATH, f"{x}.jpg"))

train_df['label_str'] = train_df['breed'].astype(str)
val_df['label_str'] = val_df['breed'].astype(str)
test_df['label_str'] = test_df['breed'].astype(str)

batch_size = 8

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    preprocessing_function=lambda x: (x - mean) / std
)

val_test_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=lambda x: (x - mean) / std
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filepath',
    y_col='label_str',
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='filepath',
    y_col='label_str',
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col='label_str',
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

print(f"✓ Batch size: {batch_size}")
print(f"✓ Train batches: {len(train_generator)}")

Found 7155 validated image filenames belonging to 120 classes.
Found 1533 validated image filenames belonging to 120 classes.
Found 1534 validated image filenames belonging to 120 classes.
✓ Batch size: 8
✓ Train batches: 895


In [5]:
def residual_block(x, filters, stride=1, conv_shortcut=False):
    """
    Basic Residual Block with skip connection
    
    The key innovation: output = F(x) + x
    where F(x) is the residual function (conv layers)
    """
    
    shortcut = x
    
    # First conv layer
    x = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Second conv layer
    x = layers.Conv2D(filters, 3, strides=1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    
    # Adjust shortcut if dimensions changed
    if conv_shortcut:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    
    # Key step: Add skip connection
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    return x


def create_resnet18(input_shape=(224, 224, 3), num_classes=120):
    """
    ResNet-18 architecture from scratch
    
    Structure:
    - Initial conv layer
    - 4 residual layers (each with 2 residual blocks)
    - Global average pooling
    - Fully connected layer
    """
    
    inputs = layers.Input(shape=input_shape)
    
    # Initial convolutional layer
    x = layers.Conv2D(64, 7, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(3, strides=2, padding='same')(x)
    
    # Residual Layer 1 (64 filters, 2 blocks)
    x = residual_block(x, 64, stride=1, conv_shortcut=False)
    x = residual_block(x, 64, stride=1, conv_shortcut=False)
    
    # Residual Layer 2 (128 filters, 2 blocks)
    x = residual_block(x, 128, stride=2, conv_shortcut=True)
    x = residual_block(x, 128, stride=1, conv_shortcut=False)
    
    # Residual Layer 3 (256 filters, 2 blocks)
    x = residual_block(x, 256, stride=2, conv_shortcut=True)
    x = residual_block(x, 256, stride=1, conv_shortcut=False)
    
    # Residual Layer 4 (512 filters, 2 blocks)
    x = residual_block(x, 512, stride=2, conv_shortcut=True)
    x = residual_block(x, 512, stride=1, conv_shortcut=False)
    
    # Global average pooling and fully connected
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs, outputs, name='resnet18')
    
    return model


# Create model
model = create_resnet18(input_shape=(img_size, img_size, 3), num_classes=num_classes)

# Count parameters
total_params = model.count_params()

print(f"\n✓ Model: ResNet-18 (from scratch)")
print(f"✓ Total parameters: {total_params:,}")
print(f"✓ Number of classes: {num_classes}")

# Print model summary
print("\nModel Architecture Summary:")
model.summary()


✓ Model: ResNet-18 (from scratch)
✓ Total parameters: 11,247,672
✓ Number of classes: 120

Model Architecture Summary:
Model: "resnet18"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 112, 112, 64  9408        ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 batch_normalization (BatchNorm  (None, 112, 112, 64  256        ['con

In [6]:
learning_rate = 0.001
num_epochs = 25

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"✓ Optimizer: Adam")
print(f"✓ Learning rate: {learning_rate}")
print(f"✓ Loss function: Categorical Crossentropy")
print(f"✓ Epochs: {num_epochs}")

✓ Optimizer: Adam
✓ Learning rate: 0.001
✓ Loss function: Categorical Crossentropy
✓ Epochs: 25


In [ ]:
callbacks = [
    ModelCheckpoint(
        'resnet_scratch_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    )
]

print("✓ Callbacks configured")

✓ Callbacks configured


: 

In [ ]:
history = model.fit(
    train_generator,
    epochs=num_epochs,
    validation_data=val_generator,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

training_time = time.time()

print("\n" + "=" * 70)
print("TRAINING COMPLETED!")

Epoch 1/25


In [ ]:
history_df = pd.DataFrame({
    'epoch': range(1, len(history.history['loss']) + 1),
    'train_loss': history.history['loss'],
    'train_acc': [acc * 100 for acc in history.history['accuracy']],
    'val_loss': history.history['val_loss'],
    'val_acc': [acc * 100 for acc in history.history['val_accuracy']],
    'lr': history.history['lr']
})

history_df.to_csv('resnet_scratch_history.csv', index=False)
print("✓ Saved: resnet_scratch_history.csv")

best_epoch = history_df['val_acc'].idxmax() + 1
best_val_acc = history_df['val_acc'].max()

In [ ]:
model = keras.models.load_model('resnet_scratch_best.h5')
print("✓ Loaded best model")

test_loss, test_acc = model.evaluate(test_generator, verbose=0)
test_acc *= 100

print(f"\nTest Set Results:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.2f}%")

# Calculate Top-5 Accuracy
print("\nCalculating Top-5 Accuracy...")
test_generator.reset()
y_true = test_generator.classes
y_pred_probs = model.predict(test_generator, verbose=1)

# Top-5 accuracy
top5_correct = 0
for i in range(len(y_true)):
    top5_preds = np.argsort(y_pred_probs[i])[-5:]
    if y_true[i] in top5_preds:
        top5_correct += 1

top5_accuracy = (top5_correct / len(y_true)) * 100
print(f"  Top-5 Accuracy: {top5_accuracy:.2f}%")

In [ ]:
print("\n[9] Generating Visualizations...")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss curves
axes[0].plot(history_df['epoch'], history_df['train_loss'], 
             label='Train Loss', linewidth=2)
axes[0].plot(history_df['epoch'], history_df['val_loss'], 
             label='Val Loss', linewidth=2)
axes[0].axvline(x=best_epoch, color='red', linestyle='--', 
                label=f'Best Epoch ({best_epoch})')
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Loss', fontweight='bold')
axes[0].set_title('ResNet-18 (Scratch) - Loss Curves', fontweight='bold', fontsize=14)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy curves
axes[1].plot(history_df['epoch'], history_df['train_acc'], 
             label='Train Acc', linewidth=2)
axes[1].plot(history_df['epoch'], history_df['val_acc'], 
             label='Val Acc', linewidth=2)
axes[1].axvline(x=best_epoch, color='red', linestyle='--', 
                label=f'Best Epoch ({best_epoch})')
axes[1].axhline(y=best_val_acc, color='green', linestyle=':', 
                label=f'Best Val Acc ({best_val_acc:.2f}%)')
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Accuracy (%)', fontweight='bold')
axes[1].set_title('ResNet-18 (Scratch) - Accuracy Curves', fontweight='bold', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('resnet_scratch_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: resnet_scratch_training_curves.png")


In [ ]:
try:
    baseline_history = pd.read_csv('baseline_cnn_history.csv')
    with open('baseline_cnn_summary.json', 'r') as f:
        baseline_summary = json.load(f)
    
    # Create comparison plot
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Validation Loss comparison
    axes[0].plot(baseline_history['epoch'], baseline_history['val_loss'], 
                 label='Baseline CNN', linewidth=2, alpha=0.8)
    axes[0].plot(history_df['epoch'], history_df['val_loss'], 
                 label='ResNet-18', linewidth=2, alpha=0.8)
    axes[0].set_xlabel('Epoch', fontweight='bold')
    axes[0].set_ylabel('Validation Loss', fontweight='bold')
    axes[0].set_title('Validation Loss Comparison', fontweight='bold', fontsize=14)
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Validation Accuracy comparison
    axes[1].plot(baseline_history['epoch'], baseline_history['val_acc'], 
                 label='Baseline CNN', linewidth=2, alpha=0.8)
    axes[1].plot(history_df['epoch'], history_df['val_acc'], 
                 label='ResNet-18', linewidth=2, alpha=0.8)
    axes[1].set_xlabel('Epoch', fontweight='bold')
    axes[1].set_ylabel('Validation Accuracy (%)', fontweight='bold')
    axes[1].set_title('Validation Accuracy Comparison', fontweight='bold', fontsize=14)
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('resnet_vs_baseline_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: resnet_vs_baseline_comparison.png")
    
    # Print comparison
    print("\nPerformance Comparison:")
    print(f"{'Metric':<25} {'Baseline CNN':>15} {'ResNet-18':>15} {'Improvement':>15}")
    print("-" * 75)
    print(f"{'Best Val Accuracy':<25} {baseline_summary['best_val_acc']:>14.2f}% {best_val_acc:>14.2f}% {best_val_acc - baseline_summary['best_val_acc']:>14.2f}%")
    print(f"{'Test Accuracy':<25} {baseline_summary['test_acc']:>14.2f}% {test_acc:>14.2f}% {test_acc - baseline_summary['test_acc']:>14.2f}%")
    print(f"{'Parameters':<25} {baseline_summary['total_params']:>15,} {total_params:>15,} {total_params - baseline_summary['total_params']:>15,}")
    print(f"{'Training Time (min)':<25} {baseline_summary['training_time_minutes']:>14.2f}m {training_time/60:>14.2f}m {(training_time/60) - baseline_summary['training_time_minutes']:>14.2f}m")
    
except FileNotFoundError:
    print("⚠ Baseline results not found. Run 03_baseline_cnn.ipynb first for comparison.")

In [ ]:
print("RESNET-18 (FROM SCRATCH) SUMMARY")
print("=" * 70)

summary = {
    'model': 'ResNet-18 (from scratch)',
    'total_params': int(total_params),
    'trainable_params': int(total_params),
    'epochs_trained': len(history_df),
    'best_epoch': int(best_epoch),
    'best_val_acc': float(best_val_acc),
    'test_acc': float(test_acc),
    'top5_acc': float(top5_accuracy),
    'training_time_minutes': float(training_time / 60),
    'final_train_acc': float(history_df['train_acc'].iloc[-1]),
    'final_val_acc': float(history_df['val_acc'].iloc[-1])
}

for key, value in summary.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

# Save summary
with open('resnet_scratch_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("\n✓ Saved: resnet_scratch_summary.json")